# Implementação da Arquitetura LeNet-5

Este notebook contém a implementação da clássica arquitetura LeNet-5 utilizando PyTorch. Esta rede foi proposta por Yann LeCun et al. em 1998 para o reconhecimento de dígitos manuscritos.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        # C1: 1 input image channel, 6 output channels, 5x5 square convolution
        # we add padding=2 to handle 28x28 MNIST images like the original 32x32 input
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2)
        # S2: Max pooling layer with 2x2 kernel and stride 2
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # C3: 6 input channels, 16 output channels, 5x5 square convolution
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # S4: Max pooling layer with 2x2 kernel and stride 2
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # C5: 16 input channels, 120 output channels, 5x5 square convolution
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5)
        # F6: Fully connected layer
        self.fc1 = nn.Linear(120, 84)
        # Output level: Fully connected layer
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        # Apply C1 followed by ReLU activation and S2 pooling
        x = self.pool1(F.relu(self.conv1(x)))
        # Apply C3 followed by ReLU activation and S4 pooling
        x = self.pool2(F.relu(self.conv2(x)))
        # Apply C5 followed by ReLU activation
        x = F.relu(self.conv3(x))
        # Flatten the output for the fully connected layers
        x = x.view(x.size(0), -1)
        # Apply F6 followed by ReLU activation
        x = F.relu(self.fc1(x))
        # Final output layer
        x = self.fc2(x)
        return x

## Visualização dos Filtros (Pesos Iniciais)

Abaixo, instanciamos o modelo e visualizamos os pesos aleatórios da primeira camada convolucional (`conv1`) antes de qualquer treinamento.

In [3]:
import matplotlib.pyplot as plt

# Instantiate the model
model = LeNet5(num_classes=10)

# Extract filters from the first convolutional layer
filters = model.conv1.weight.detach()

# Plotting the filters
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i, ax in enumerate(axes):
    # Standard convolution filters have depth (channels); we take the first slice
    # Since MNIST is grayscale, input_channels = 1
    ax.imshow(filters[i, 0], cmap='gray')
    ax.set_title(f'Filter {i+1}')
    ax.axis('off')

plt.suptitle('Initialization State of LeNet-5 Conv1 Filters')
plt.show()

## Carregamento e Visualização do Dataset MNIST

Agora vamos carregar o dataset MNIST utilizando `torchvision` e visualizar algumas imagens de exemplo.

In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transform to convert images to tensors
transform = transforms.Compose([transforms.ToTensor()])

# Download and load training data
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True)

# Get some random images
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Plot images
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {labels[i].item()}')
    ax.axis('off')

plt.suptitle('Sample Images from MNIST Dataset')
plt.show()

## Treinamento do Modelo

Nesta seção, definimos o loop de treinamento. Utilizaremos o otimizador Adam e a função de perda CrossEntropyLoss para treinar nossa LeNet-5 no dataset MNIST.

In [5]:
import torch.optim as optim

# Device configuration (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Hyperparameters
learning_rate = 0.001
batch_size = 64
num_epochs = 5

# Update DataLoader with larger batch size for training
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
print(f'Training on {device}...')
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

print("Training finished!")

Training on cpu...
Epoch [1/5], Step [100/938], Loss: 0.6579
Epoch [1/5], Step [200/938], Loss: 0.1826
Epoch [1/5], Step [300/938], Loss: 0.1475
Epoch [1/5], Step [400/938], Loss: 0.1356
Epoch [1/5], Step [500/938], Loss: 0.0748
Epoch [1/5], Step [600/938], Loss: 0.0771
Epoch [1/5], Step [700/938], Loss: 0.1403
Epoch [1/5], Step [800/938], Loss: 0.1451
Epoch [1/5], Step [900/938], Loss: 0.1264
Epoch [2/5], Step [100/938], Loss: 0.1255
Epoch [2/5], Step [200/938], Loss: 0.0131
Epoch [2/5], Step [300/938], Loss: 0.0478
Epoch [2/5], Step [400/938], Loss: 0.0501
Epoch [2/5], Step [500/938], Loss: 0.0156
Epoch [2/5], Step [600/938], Loss: 0.0061
Epoch [2/5], Step [700/938], Loss: 0.0576
Epoch [2/5], Step [800/938], Loss: 0.0608
Epoch [2/5], Step [900/938], Loss: 0.0463
Epoch [3/5], Step [100/938], Loss: 0.0985
Epoch [3/5], Step [200/938], Loss: 0.0197
Epoch [3/5], Step [300/938], Loss: 0.0314
Epoch [3/5], Step [400/938], Loss: 0.0349
Epoch [3/5], Step [500/938], Loss: 0.0140
Epoch [3/5], St